# GP03 CPU Qualification

Qualify public test execution and recover environment-specific holds.

This source copy comes from **GP03_CPU_Qualification_Retry_v1_1.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

# GP03 CPU qualification retry v1.1

This standalone retry retains the 10 ready cases and two no-match holds from v1.0. It revisits 24 cases affected by archive staging, pytest node paths or missing dependencies.

1. Import this notebook into Kaggle. Use the same CPU runtime as v1.0, with **Accelerator None** and **Internet Off**.
2. Add the **Gemma 4 Developer Agent** competition input. The expected mount is `/kaggle/input/competitions/gemma-4-developer-agent`.
3. Make the original `GP03_CPU_Qualification_20261108T223352Z_0165d358.zip` available in `/kaggle/working` or as a mounted input. The notebook checks its checksum and CPU environment fingerprint before carrying results forward.
4. Choose **Run All**. Download the new `GP03_CPU_Qualification_*.zip` from the final cell or the Output panel.

The output reports all 36 GP00 cases and labels retained and new results separately. Model calls and protected grades are zero. The 74% checkpoint is the later evaluation freeze.

Each task has a saved checkpoint under `/kaggle/working/GP03_CPU_Checkpoints`. Keep that directory and the v1.0 ZIP available to resume an interrupted run.


## 1. Kaggle settings

In [ ]:
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path, PurePosixPath
from typing import Mapping, Sequence
import ast
import csv
import fcntl
import hashlib
import json
import os
import platform
import posixpath
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import time
import uuid
import xml.etree.ElementTree as ET
import zipfile

COMPETITION_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WORK_DIR = Path("/kaggle/working")
RUN_LABEL = "retry_v1_1"
AUTO_DOWNLOAD_ZIP = True
PRIOR_CPU_ZIP = None  # Optional explicit path to the original v1.0 evidence ZIP
PRIOR_ZIP_NAME = "GP03_CPU_Qualification_20261108T223352Z_0165d358.zip"
PRIOR_ZIP_SHA256 = "e388230dc53422604b2ea57e05ed261b7d01920f0536436dadf7b935c91c7378"
PRIOR_RECIPE_SHA256 = "41d21da3db3cc5dca8147cc799ce02b4feb838e3c3c66244a94616c5d8d81701"
GP03_PRIOR = {"name": PRIOR_ZIP_NAME, "sha256": PRIOR_ZIP_SHA256, "recipe": PRIOR_RECIPE_SHA256}

POLICY = {
    "version": "GP03_CPU_Qualification_v1.1",
    "selector_rule": "public_issue_to_tracked_test_paths_and_symbols_v1",
    "max_selectors": 5,
    "large_file_test_functions": 50,
    "max_collected_nodes": 200,
    "snapshot_bytes_expanded_cap": 6 * 1024**3,
    "task_wall_seconds": 300,
    "collect_wall_seconds": 55,
    "baseline_wall_seconds": 40,
    "final_reserve_seconds": 60,
    "optional_import_fallback": [
        "dirty_equals", "email_validator", "httpbin", "inline_snapshot", "multipart", "orjson",
        "pytest_asyncio", "pytest_httpbin", "pytest_mock", "trustme", "ujson", "uvicorn",
    ],
    "optional_fixture_fallback": ["httpbin", "httpbin_secure"],
}
PACKAGE_LAYOUTS = {
    "fastapi/fastapi": ("fastapi", ("fastapi", "src/fastapi")),
    "Textualize/rich": ("rich", ("rich", "src/rich")),
    "psf/requests": ("requests", ("requests", "src/requests")),
}
EXTRA_DEPENDENCIES = (
    "attrs", "brotli", "charset-normalizer", "cryptography", "dirty-equals",
    "flask", "httpbin", "pytest-httpbin", "pytest-mock", "requests-mock",
    "simplejson", "socksio", "trustme", "uvicorn", "werkzeug",
)
OFFLINE_FLAGS = {
    "CUDA_VISIBLE_DEVICES": "",
    "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_OPTIONAL_LOCKS": "0",
    "GIT_TERMINAL_PROMPT": "0",
    "HF_HUB_OFFLINE": "1",
    "PIP_CONFIG_FILE": os.devnull,
    "PIP_NO_INDEX": "1",
    "PIP_NO_INPUT": "1",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONHASHSEED": "0",
    "PYTHONNOUSERSITE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
PUBLIC_TEST = re.compile(r"tests/(?:[A-Za-z0-9_-]+/)*[A-Za-z0-9_-]+\.py")
VALID_ID = re.compile(r"[A-Za-z0-9_]+")
GIT_ID = re.compile(r"[0-9a-f]{40}")


class CPUHold(RuntimeError):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason = reason

## 2. Provisional cohort

The list contains one candidate per repository and base-commit group. The eight development IDs are included for the separation check.


In [ ]:
NOTEBOOK_RECIPE_SHA256 = '0ae4bc06d88271e9b2ccd3c4fcdae182d9b58b07743eb7130cf7daa317a0f750'

TASKS_SHA256 = 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6'
GP00_SPLIT_SHA256 = '27b50b7f46c510c55b7b7bb264aba729abba0cc9e53647e596690c94cd79bb09'

# task ID, repository, organizer base commit, GP00 archive size in bytes
GP03_CANDIDATE_ROWS = [
    ('fastapi_14964', 'fastapi/fastapi', 'bc06e4296d588f39f38c7ac2fc96d3913c565962', 271065089),
    ('fastapi_15023', 'fastapi/fastapi', 'c0836dc1b7f4ba91e9563c8c4ccd0ebe655bc914', 275792435),
    ('requests_7505', 'psf/requests', '6f205ff422bccd5e4c4fc0b64c5f3e7df5181db6', 37499757),
    ('requests_7502', 'psf/requests', '661970d171d9c3e12e4c789c4768db647d8c4da0', 37463957),
    ('requests_7433', 'psf/requests', '0b401c76b6e80a4eecf3c690085b2553f6e261ca', 37233814),
    ('requests_7427', 'psf/requests', 'b684dcb9bbf3aa557d1238e72062c4a29737dd1c', 37176965),
    ('requests_7315', 'psf/requests', '6360477c52303c9445b45fa8744b02d05a2f0905', 37602543),
    ('requests_7328', 'psf/requests', 'cbce031327be4f1b4b5fd041ff4dcaa8efa2ce53', 37608316),
    ('requests_7309', 'psf/requests', '7407309c8a8a73aa2f4337184025d440bbedab7a', 37530773),
    ('requests_7205', 'psf/requests', '1b40fdd004bfc8ba5301bcf8a6908264e9b6b877', 37402474),
    ('rich_4070', 'Textualize/rich', 'fc41075a3206d2a5fd846c6f41c4d2becab814fa', 96686753),
    ('rich_4077', 'Textualize/rich', '58ac15120fe254886c10d6253667733c83604cb4', 97076310),
    ('fastapi_14258', 'fastapi/fastapi', 'e94028ab60d18080810910352bc9129e8046c92d', 259958390),
    ('fastapi_14794', 'fastapi/fastapi', '464c359bb03f7ba72330198cc9f44df36194837d', 254845131),
    ('fastapi_14463', 'fastapi/fastapi', 'f2687dc1bb01f4cb62eee90e656b61c38c2aaf6a', 240910100),
    ('fastapi_12942', 'fastapi/fastapi', '27c0f7e75fc0ce26ca75295d455eb36f0ca033d3', 218214164),
    ('fastapi_14482', 'fastapi/fastapi', '71a17b5932e38bcf798fcd05c5bb384e6d4db5bf', 230406330),
    ('fastapi_14458', 'fastapi/fastapi', '516169428d2fa189d34318ebc469a082c49c1189', 229629718),
    ('fastapi_14266', 'fastapi/fastapi', '2cf04ee30db9df2db04a5b1e0d2c625dfbf1a211', 221646802),
    ('rich_3718', 'Textualize/rich', '7d36119f8378b36129502c3281a7faffef837e02', 95464718),
    ('rich_3953', 'Textualize/rich', '1d402e0c59f8765e420a5a4440eb2fca7465d1ae', 96570158),
    ('rich_3944', 'Textualize/rich', '36fe3f7ca9becca4777861d5e6e625f5a4a37545', 96523552),
    ('fastapi_15763', 'fastapi/fastapi', '9a9c4ad5d06f5fe8ee6775a5aeaa2f83c854f263', 326855675),
    ('fastapi_15745', 'fastapi/fastapi', '016ab760faba1813ba235d5582a014a58bd2f417', 326518082),
    ('fastapi_14303', 'fastapi/fastapi', '740ec2787b25e59f7aa9e3df1db09d4489c22718', 226787652),
    ('requests_6644', 'psf/requests', '382fc2c0c6c0ef0874bc65bc1175f97c073e5086', 36385820),
    ('requests_6589', 'psf/requests', '7a13c041dbef42f9f3feb14110f02626f6892e9a', 36361445),
    ('requests_6592', 'psf/requests', '0b4d494192de489701d3a2e32acef8fb5d3f042e', 36265394),
    ('rich_3772', 'Textualize/rich', '2809954800af15b4027364cebf82390fe23dc194', 95103715),
    ('rich_3535', 'Textualize/rich', '0f2f51b872d14588de3a65968f5cdde6fb5694a3', 94267814),
    ('rich_3506', 'Textualize/rich', '4101991898ee7a09fe1706daca24af5e1e054862', 93416675),
    ('rich_3470', 'Textualize/rich', '0ba607dc080c9f102b5ae5346bcdb70d6e9c969d', 92924508),
    ('rich_3278', 'Textualize/rich', '4c5d305bb902ce97011666ada19e48d0b27b49e1', 92497686),
    ('rich_3043', 'Textualize/rich', '8a616de8e25db3c93abaaa220c493d80878de69d', 90818810),
    ('rich_3067', 'Textualize/rich', '1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d', 90917632),
    ('rich_3006', 'Textualize/rich', '0c8bb032ad3b54a32fdb8a43f5ab457c8cd6dcb1', 90866513),
]
GP03_CANDIDATES = [
    dict(task_id=task, repo=repo, base_commit=base, snapshot_bytes=size)
    for task, repo, base, size in GP03_CANDIDATE_ROWS
]

GP03_DEVELOPMENT = [
    ('fastapi_15589', 'fastapi/fastapi', '22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b'),
    ('rich_4079', 'Textualize/rich', '19c67b9a3479841e9133bea94607c89ee931d3fc'),
    ('fastapi_14479', 'fastapi/fastapi', '2b212ddd7604891d42e2ffbafe764c334230bdb7'),
    ('fastapi_5624', 'fastapi/fastapi', 'c6487ed632056e450d844846a1b63be551a3cbc6'),
    ('httpx_3672', 'encode/httpx', '4acf5c2c37714cc63b5cf71b3e284fca83c90311'),
    ('requests_6757', 'psf/requests', '0e322af87745eff34caffe4df68456ebc20d9068'),
    ('rich_2943', 'Textualize/rich', '610fd75c433c3dfcd9c296e8b91363979127b9fa'),
    ('rich_3130', 'Textualize/rich', '720800e6930d85ad027b1e9bd0cbb96b5e994ce3'),
]

## 3. Reused source, test and evidence helpers

In [ ]:
# Reused from the pinned GP02C-04B source gp02c_ledger.py.
EXCLUDED_DIRS = frozenset({
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".tox", ".nox", ".cache", "node_modules", ".ipynb_checkpoints",
    ".gp02c_results", "__ledger_output__", "htmlcov", ".coverage-data",
})

EXCLUDED_NAMES = frozenset({".coverage", ".DS_Store", "coverage.xml"})

EXCLUDED_SUFFIXES = frozenset({".pyc", ".pyo"})

PYTEST_CONFIG_NAMES = (
    "pytest.ini", ".pytest.ini", "pyproject.toml", "setup.cfg", "tox.ini",
    "pytest.toml", ".pytest.toml",
)

RICH_OPTIONAL_TEST_DEFINITIONS = (
    "conftest.py", "tests/conftest.py", "tests/__init__.py",
    *PYTEST_CONFIG_NAMES, *(f"tests/{name}" for name in PYTEST_CONFIG_NAMES),
)

def canonical_hash(obj: object) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=True,
                                      separators=(",", ":"), default=str).encode()).hexdigest()

def environment_fingerprint(config: Mapping[str, object]) -> str:
    """Hash explicit environment identity supplied by the trusted harness.

    In live integration, include installed wheel/source hashes and relevant runtime
    settings, not just package names, and refresh after environment modifications.
    """
    if not config:
        raise ValueError("Nonempty trusted environment manifest required")
    return canonical_hash(dict(config))

@dataclass(frozen=True)
class Snapshot:
    sha256: str
    files: int
    entries: tuple[tuple[str, str, str], ...]

class WorkspaceFingerprinter:
    """Conservative all-files policy, except a fixed generated-artifact exclusion list.

    Includes untracked source and config; file additions, deletions and symlink
    targets affect the fingerprint. NEVER walk an evaluator/protected workspace.
    """
    def __init__(self, root: str | Path):
        self.root = Path(root).resolve(strict=True)
        if not self.root.is_dir():
            raise ValueError("Workspace root must be a directory")

    def snapshot(self) -> Snapshot:
        entries: list[tuple[str, str, str]] = []
        for base, dirs, files in os.walk(self.root, followlinks=False):
            basedir = Path(base)
            # Symlink directories are recorded as links, never dereferenced.
            for name in list(dirs):
                full = basedir / name
                if name in EXCLUDED_DIRS:
                    dirs.remove(name)
                elif full.is_symlink():
                    dirs.remove(name)
                    rel = full.relative_to(self.root).as_posix()
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
            for name in files:
                if name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                    continue
                full = basedir / name
                rel = full.relative_to(self.root).as_posix()
                if full.is_symlink():
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
                elif full.is_file():
                    h = hashlib.sha256()
                    with full.open("rb") as f:
                        for block in iter(lambda: f.read(256 * 1024), b""):
                            h.update(block)
                    entries.append((rel, "F", h.hexdigest()))
        entries.sort()
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))

def definition_files_fingerprint(root: str | Path, relative_files: Sequence[str], *,
                                 allow_missing: bool = False) -> str:
    """Bind a selected visible test to its current test/config file bytes.

    File deletions, symlinks, path traversal and non-files fail closed. Caller
    supplies only approved visible test/config paths; NEVER protected tests.
    """
    root = Path(root).resolve(strict=True)
    if not relative_files or len(relative_files) != len(set(relative_files)):
        raise ValueError("Nonempty unique visible test definition paths required")
    entries = []
    for name in sorted(relative_files):
        relative = Path(name)
        if (not name or relative.is_absolute() or any(x in {"..", "."} for x in relative.parts)
            or relative.as_posix() != name):
            raise ValueError("Invalid or noncanonical visible definition path")
        path = root / relative
        if path.is_symlink() or not path.resolve().is_relative_to(root):
            raise ValueError("Visible definition unsafe or symlinked")
        if not path.is_file():
            if (allow_missing and not path.exists() and
                (name in RICH_OPTIONAL_TEST_DEFINITIONS or relative.name == "conftest.py")):
                entries.append((name, 'ABSENT_AT_FREEZE'))
                continue
            raise ValueError("Visible definition missing or unsafe")
        entries.append((name, hashlib.sha256(path.read_bytes()).hexdigest()))
    return canonical_hash(entries)

In [ ]:
# Reused from the pinned GP02C-04B source gp02c_dev_registry.py.
CONFIG_FILES = ('setup.py', 'setup.cfg', 'pytest.ini', '.pytest.ini', 'pytest.toml',
                '.pytest.toml', 'pyproject.toml', 'tox.ini', 'conftest.py')

TEST_DEPENDENCIES = (
    'pytest', 'pluggy', 'iniconfig', 'packaging', 'pytest-asyncio', 'pytest-cov',
    'pytest-timeout', 'coverage', 'typing-extensions', 'pydantic', 'pydantic-core',
    'typing-inspection', 'annotated-types', 'annotated-doc', 'starlette', 'anyio',
    'httpx', 'httpcore', 'h11', 'h2', 'hpack', 'hyperframe', 'idna', 'sniffio',
    'certifi', 'requests', 'urllib3', 'python-multipart', 'email-validator',
    'markdown-it-py', 'mdurl', 'pygments', 'exceptiongroup', 'trio',
    'fastapi-cli', 'jinja2', 'orjson', 'ujson', 'pyyaml',
)

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    temporary.replace(path)

In [ ]:
# Reused from the pinned GP02C-04B source gp02c_live_adapter.py.
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(262144), b""):
            h.update(block)
    return h.hexdigest()

def junit_counts(junit: Path | bytes) -> tuple[int, int, int, int, int]:
    """Extract actual testcase elements; never trust exit code or suite labels alone."""
    data = junit if isinstance(junit, bytes) else junit.read_bytes()
    root = ET.fromstring(data)
    cases = [el for el in root.iter() if el.tag.rsplit("}", 1)[-1] == "testcase"]
    collected = len(cases)
    fail = sum(any(c.tag.rsplit("}", 1)[-1] == "failure" for c in t) for t in cases)
    err = sum(any(c.tag.rsplit("}", 1)[-1] == "error" for c in t) for t in cases)
    skipped = sum(any(c.tag.rsplit("}", 1)[-1] == "skipped" for c in t) for t in cases)
    passed = collected - fail - err - skipped
    if passed < 0 or collected != fail + err + skipped + passed:
        raise ValueError("Contradictory JUnit testcase counts")
    return collected, passed, fail, err, skipped

def _cancel_process_group(proc: subprocess.Popen, grace: float = 0.15) -> None:
    """Terminate a process GROUP even if its leader has already exited.

    A parent exiting after SIGTERM does not prove children have exited. All
    checks run in their own POSIX session. After a brief grace, SIGKILL is sent
    to the *group*, including remaining descendants, even if leader.poll() != None.
    """
    if os.name == "posix":
        pgid = proc.pid  # subprocess.Popen(start_new_session=True)
        try:
            os.killpg(pgid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=max(grace, 0.02))
        except subprocess.TimeoutExpired:
            pass
        # Crucial: don't return merely because the group LEADER exited.
        # An orphan/daemonized child in this process group may still be alive.
        time.sleep(min(0.04, max(grace, 0.0)))
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=2)
        except subprocess.TimeoutExpired:
            pass
    else:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=2)

In [ ]:
# Reused from the pinned GP02C-04B source gp02c_dev_preflight.py.
HERMETIC_INI = """[pytest]
addopts = -p no:anyio
norecursedirs = .* build dist venv
python_classes = Test* *Test
python_files = test_*.py *_test.py
filterwarnings =
    ignore::DeprecationWarning
    ignore::UserWarning
"""

HERMETIC_HEADER = '# Hermetic test discovery hook for SWE-gemma\n'

def seal_zip(output, files):
    """Write a new evidence archive and index every included file."""
    output = Path(output)
    index = {}
    temporary = output.with_suffix('.partial.zip')
    if output.exists() or temporary.exists():
        raise FileExistsError(f'Refusing to overwrite evidence {output}')
    try:
        with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_DEFLATED) as z:
            for name, path in sorted(files.items()):
                path = Path(path)
                if path.is_symlink() or not path.is_file():
                    raise ValueError(f'Unsafe evidence file {name}')
                index[name] = {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
                z.write(path, name)
            z.writestr('SHA256_MANIFEST.json', json.dumps(index, indent=2, sort_keys=True) + '\n')
        verify_zip(temporary)
        temporary.rename(output)
    finally:
        temporary.unlink(missing_ok=True)
    print(f'Evidence ZIP {output.name} | SHA-256 {sha256_file(output)}', flush=True)
    return output

def verify_zip(archive):
    with zipfile.ZipFile(archive) as z:
        names = z.namelist()
        index = json.loads(z.read('SHA256_MANIFEST.json'))
        if len(names) != len(set(names)) or set(names) != set(index) | {'SHA256_MANIFEST.json'}:
            raise RuntimeError('ZIP member set disagrees with the checksum manifest')
        for name, pin in index.items():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts or z.getinfo(name).file_size != pin['bytes']:
                raise RuntimeError(f'Invalid archive member {name}')
            digest = hashlib.sha256()
            with z.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != pin['sha256']:
                raise RuntimeError(f'Archive checksum mismatch for {name}')
    return index

In [ ]:
# Reused from the pinned GP02C-04B source gp02c_dev_browser_download.py.
def offer_zip_download(bundle, auto_download=True, embed_limit_mb=10):
    from pathlib import Path
    import base64
    import html
    import json
    import os
    from IPython.display import FileLink, HTML, Javascript, display

    bundle = Path(bundle).resolve()
    relative = os.path.relpath(bundle, Path.cwd())
    print('ZIP ready:', bundle)
    print('If the browser does not start a download, use the Download ZIP button or Kaggle Output panel.')
    display(FileLink(relative))
    if bundle.stat().st_size > embed_limit_mb * 1024 * 1024:
        print('This ZIP exceeds the inline-download size limit; use the file link or Output panel.')
        return {'automatic_download_requested': False, 'download_confirmed': False, 'reason': 'size_limit'}

    encoded = base64.b64encode(bundle.read_bytes()).decode('ascii')
    name = html.escape(bundle.name, quote=True)
    button = ('<a download="' + name + '" href="data:application/zip;base64,' + encoded + '" '
              'style="display:inline-block;padding:12px 20px;background:#174EA6;color:white;'
              'text-decoration:none;border-radius:6px;font-weight:600">Download ZIP</a>')
    display(HTML(button))
    if auto_download:
        script = '''(function () {
  try {
    const payload = __PAYLOAD__;
    const bytes = Uint8Array.from(atob(payload), function (c) { return c.charCodeAt(0); });
    const url = URL.createObjectURL(new Blob([bytes], {type: "application/zip"}));
    const link = document.createElement("a");
    link.href = url;
    link.download = __FILENAME__;
    link.style.display = "none";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(function () { URL.revokeObjectURL(url); }, 60000);
  } catch (error) {
    console.info("Automatic ZIP download was not available; use the Download ZIP button or Output panel.", error);
  }
})();'''.replace('__PAYLOAD__', json.dumps(encoded)).replace('__FILENAME__', json.dumps(bundle.name))
        display(Javascript(script))
        print('Automatic browser-download request offered. Notebook code cannot confirm that your PC saved it.')
    return {'automatic_download_requested': bool(auto_download), 'download_confirmed': False,
            'fallbacks': ['Download ZIP button', 'file link', 'Kaggle Output panel']}

## 4. Verify the public release and record the CPU environment

In [ ]:
def load_gp03_inputs(competition, candidates, development, release_sha):
    """Use issue text and metadata from the pinned release, with an explicit projection."""
    if not (competition / "tasks.jsonl").is_file():
        raise CPUHold("TASK_RELEASE_MISSING", f"Mount the Developer Agent input at {competition}")
    if not (competition / "snapshots").is_dir():
        raise CPUHold("SNAPSHOT_MOUNT_MISSING", f"Expected {competition / 'snapshots'}")
    if sha256_file(competition / "tasks.jsonl") != release_sha:
        raise CPUHold("TASK_RELEASE_MISMATCH", "tasks.jsonl SHA-256 differs from the GP00 release")
    index = {r["task_id"]: r for r in candidates}
    groups = {(r["repo"], r["base_commit"]) for r in candidates}
    dev_ids = {r[0] for r in development}
    dev_groups = {(r[1], r[2]) for r in development}
    if len(index) != len(candidates) or len(groups) != len(candidates):
        raise CPUHold("COHORT_DUPLICATE", "Candidate IDs and repository/commit groups must be unique")
    if dev_ids.intersection(index) or dev_groups.intersection(groups):
        raise CPUHold("COHORT_OVERLAP", "Candidate set intersects the development set")
    for task, row in index.items():
        if (not VALID_ID.fullmatch(task) or row["repo"] not in PACKAGE_LAYOUTS
                or not GIT_ID.fullmatch(row["base_commit"]) or row["snapshot_bytes"] <= 0):
            raise CPUHold("COHORT_INVALID", f"Invalid public identity for {task}")

    issues = {}
    with (competition / "tasks.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            raw = json.loads(line)
            task = raw.get("instance_id")
            if task in index:
                pin = index[task]
                if task in issues or (raw.get("repo"), raw.get("base_commit")) != (
                    pin["repo"], pin["base_commit"]
                ):
                    raise CPUHold("TASK_IDENTITY_MISMATCH", f"Release identity differs for {task}")
                issue = raw.get("problem_statement")
                if not isinstance(issue, str) or not issue.strip():
                    raise CPUHold("PUBLIC_ISSUE_MISSING", f"Public issue description missing for {task}")
                issues[task] = issue
            del raw
    if set(issues) != set(index):
        raise CPUHold("CANDIDATE_MISSING_FROM_RELEASE", "A selected GP00 case is missing")
    return issues


def environment_profile():
    if os.name != "posix" or sys.version_info < (3, 12):
        raise CPUHold("RUNTIME_UNSUPPORTED", "Use the Kaggle Linux CPU image with Python 3.12 or later")
    try:
        import pytest
    except ImportError as exc:
        raise CPUHold("PYTEST_UNAVAILABLE", "Use the organizer CPU image with pytest available") from exc
    if shutil.which("git") is None:
        raise CPUHold("GIT_UNAVAILABLE", "The published snapshots require Git")
    packages = {}
    for name in sorted(set(TEST_DEPENDENCIES) | set(EXTRA_DEPENDENCIES)):
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
        else:
            record = dist.read_text("RECORD") or dist.read_text("METADATA") or ""
            packages[name] = {"version": dist.version, "record_sha256": hashlib.sha256(record.encode()).hexdigest()}
    return {
        "python_version": platform.python_version(),
        "python_executable": sys.executable,
        "python_executable_sha256": sha256_file(Path(sys.executable).resolve()),
        "machine": platform.machine(),
        "pytest_version": pytest.__version__,
        "test_dependencies": packages,
        "flags": OFFLINE_FLAGS,
    }


def command_env(root, temporary, capture=None, package=None):
    home = temporary / "home"
    home.mkdir(exist_ok=True)
    paths = [root, root / "src"]
    if capture is not None:
        paths.append(temporary)
    env = {
        "PATH": os.environ.get("PATH", os.defpath), "HOME": str(home),
        "LANG": "C.UTF-8", "LC_ALL": "C.UTF-8", "TZ": "UTC",
        "TMPDIR": str(temporary), "TMP": str(temporary), "TEMP": str(temporary),
        "PYTHONPATH": os.pathsep.join(map(str, paths)), **OFFLINE_FLAGS,
    }
    if capture is not None:
        env.update(GP03_CAPTURE_PATH=str(capture), GP03_PUBLIC_PACKAGE=package)
    return env


def bounded_command(argv, root, env, output, timeout):
    output.mkdir(parents=True, exist_ok=False)
    started = time.monotonic()
    with (output / "stdout.txt").open("wb") as stdout, (output / "stderr.txt").open("wb") as stderr:
        proc = subprocess.Popen(
            argv, cwd=root, env=env, stdout=stdout, stderr=stderr, start_new_session=True,
        )
        timed_out = False
        try:
            proc.wait(timeout=max(0.01, timeout))
        except subprocess.TimeoutExpired:
            timed_out = True
        finally:
            try:
                os.killpg(proc.pid, 0)
            except ProcessLookupError:
                pass
            else:
                _cancel_process_group(proc)
    return {
        "argv": list(argv), "exit_code": proc.returncode, "timed_out": timed_out,
        "seconds": round(time.monotonic() - started, 3),
    }


def remaining(deadline, phase_cap):
    seconds = deadline - time.monotonic()
    if seconds <= 0:
        raise CPUHold("TASK_DEADLINE", "The CPU task wall clock expired")
    return min(seconds, phase_cap)

## 5. Rank public test selectors

Stage ordinary files before validating in-snapshot symbolic links and hardlinks. Rank tracked test files with the same issue-based policy as v1.0. Save each ordered shortlist before collection. A declared missing import or HTTP fixture can advance to the next selector before baseline execution.


In [ ]:
def snapshot_link_target(name, linkname, symbolic):
    if not linkname or PurePosixPath(linkname).is_absolute() or "\\" in linkname:
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Invalid target for {name}: {linkname[:180]}")
    base = name.parent.as_posix() if symbolic else "."
    target = PurePosixPath(posixpath.normpath(posixpath.join(base, linkname)))
    if ".." in target.parts or target == PurePosixPath("."):
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Target escapes or points to the archive root: {name}")
    return target


def install_snapshot_links(destination, links, regular):
    # Ordinary files are extracted first, so archive contents cannot be written through a link.
    for name, entry in links.items():
        path = destination / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() or path.is_symlink():
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link collides with a staged file or directory: {name}")
        if entry["kind"] == "hardlink":
            target = entry["target"]
            visited = {name}
            while target in links and links[target]["kind"] == "hardlink" and target not in visited:
                visited.add(target)
                target = links[target]["target"]
            if target not in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Hardlink lacks an in-archive regular target: {name}")
            os.link(destination / target, path, follow_symlinks=False)
    for name, entry in links.items():
        if entry["kind"] == "symlink":
            path = destination / name
            if path.exists() or path.is_symlink():
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Symbolic link collides with a staged path: {name}")
            path.symlink_to(entry["linkname"])
    for name, entry in links.items():
        path = destination / name
        try:
            target = path.resolve(strict=True)
        except (OSError, RuntimeError) as exc:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Broken or cyclic archive link: {name}") from exc
        if not target.is_relative_to(destination) or target in path.parents:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Archive link target escapes or cycles to an ancestor: {name}")


def stage_public_snapshot(archive, destination, repo, deadline):
    destination = destination.resolve()
    destination.mkdir()
    total = 0
    regular, links = set(), {}
    with tarfile.open(archive, "r|gz") as tar:
        for member in tar:
            name = PurePosixPath(member.name)
            if name.is_absolute() or ".." in name.parts or "\\" in member.name:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Unsafe archive path: {member.name[:200]}")
            remaining(deadline, POLICY["task_wall_seconds"])
            if member.issym() or member.islnk():
                if name in links or name in regular or name == PurePosixPath("."):
                    raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Duplicate or invalid archive link: {name}")
                links[name] = {
                    "kind": "symlink" if member.issym() else "hardlink",
                    "target": snapshot_link_target(name, member.linkname, member.issym()),
                    "linkname": member.linkname,
                }
                continue
            if not (member.isfile() or member.isdir()) or name in links or name in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Invalid or duplicate archive member: {name} ({member.type!r})")
            total += member.size
            if total > POLICY["snapshot_bytes_expanded_cap"]:
                raise CPUHold("SNAPSHOT_SIZE_LIMIT", "Expanded archive exceeded its CPU staging limit")
            tar.extract(member, destination, filter="data")
            if member.isfile():
                regular.add(name)
    install_snapshot_links(destination, links, regular)

    package, layouts = PACKAGE_LAYOUTS[repo]
    candidates = [
        (item.parent, layout) for item in destination.rglob(".git") if item.is_dir()
        for layout in layouts if (item.parent / layout / "__init__.py").is_file()
    ]
    if len(candidates) != 1:
        raise CPUHold("SNAPSHOT_LAYOUT", f"Expected one Git checkout containing {package}")
    root, package_dir = candidates[0]
    link_inventory = []
    for name, entry in sorted(links.items()):
        path = destination / name
        target = path if entry["kind"] == "symlink" else destination / entry["target"]
        if path.is_relative_to(root) and not target.resolve(strict=True).is_relative_to(root):
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link in the checkout points outside it: {name}")
        link_inventory.append({"path": name.as_posix(), "kind": entry["kind"],
                               "target": entry["target"].as_posix()})
    return root, package, package_dir, link_inventory


def git_bytes(root, env, deadline, *argv):
    result = subprocess.run(
        ("git", "-C", str(root), *argv), env=env, capture_output=True,
        timeout=remaining(deadline, 20), check=False,
    )
    if result.returncode:
        raise CPUHold("NATIVE_GIT_PROBE", "Git could not read the mounted checkout identity")
    return result.stdout


def prepare_pytest_config(root):
    """Match the organizer subprocess harness's baseline pytest setup."""
    ini = root / "pytest.ini"
    text = ini.read_text(errors="replace") if ini.is_file() else ""
    if "# Hermetic test discovery" not in text and "[pytest]" not in text:
        ini.write_text(HERMETIC_INI, encoding="utf-8")
    conftest = root / "conftest.py"
    old = conftest.read_text(errors="replace") if conftest.is_file() else ""
    if not any(f"# {kind} test discovery hook for SWE-gemma" in old for kind in ("Hermetic", "Standard")):
        clean = old.replace(HERMETIC_HEADER, "").strip()
        conftest.write_text(HERMETIC_HEADER + ("\n" + clean if clean else "") + "\n", encoding="utf-8")


STOP_TOKENS = frozenset("""
    about actual added after again all and any api app are before been being bug can cannot
    change check class code copy could current description did does doing done error example
    expected failing fastapi feature file first fix follow found from function get got has have
    how import include instance instead issue latest like line main make markdown method more
    most must need new none not now only other out package please post problem pytest python
    read related repo reported reproducible reproduce request requests result return rich run
    same see should some source steps test tests than that the their then there these they this
    through title too trace traceback true type update use used using value version want was
    way were what when where which while will with work would you your
""".split())


def topic_tokens(text):
    text = re.sub(r"([A-Z])([A-Z][a-z])", r"\1 \2", text)
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    tokens = set()
    for token in re.findall(r"[a-z][a-z0-9]+", text.lower()):
        if len(token) > 4 and token.endswith("ies"):
            token = token[:-3] + "y"
        elif len(token) > 3 and token.endswith("s") and not token.endswith(("ss", "us", "is")):
            token = token[:-1]
        if len(token) >= 3 and token not in STOP_TOKENS:
            tokens.add(token)
    return tokens


def test_functions(path):
    try:
        tree = ast.parse(path.read_bytes())
    except (SyntaxError, UnicodeError, ValueError):
        return []
    nodes = []
    functions = (ast.FunctionDef, ast.AsyncFunctionDef)
    for node in tree.body:
        if isinstance(node, functions) and node.name.startswith("test_"):
            nodes.append(node.name)
        if isinstance(node, ast.ClassDef) and (node.name.startswith("Test") or node.name.endswith("Test")):
            nodes.extend(
                f"{node.name}::{item.name}" for item in node.body
                if isinstance(item, functions) and item.name.startswith("test_")
            )
    return nodes


def rank_public_selectors(root, tracked, issue):
    """Rank existing public definitions before any collection or execution."""
    available = []
    for name in sorted(set(tracked)):
        path = root / name
        if not PUBLIC_TEST.fullmatch(name) or not path.is_file() or path.is_symlink():
            continue
        if not path.stem.startswith("test_") and not path.stem.endswith("_test"):
            continue
        functions = test_functions(path)
        path_terms = topic_tokens(name.removesuffix(".py"))
        symbol_terms = topic_tokens(" ".join(functions))
        available.append((name, functions, path_terms, symbol_terms))
    frequency = Counter(term for _, _, p, s in available for term in p | s)
    issue_terms = topic_tokens(issue)
    code_terms = topic_tokens(" ".join(re.findall(r"`([^`\n]{1,160})`", issue)))
    issue_lower = issue.lower()

    def weight(terms):
        return sum(
            (1 + min(7, len(available) // frequency[term])) * (2 if term in code_terms else 1)
            for term in terms & issue_terms if frequency[term]
        )

    results = []
    for name, functions, path_terms, symbol_terms in available:
        exact = name.lower() in issue_lower or PurePosixPath(name).name.lower() in issue_lower
        overlap = (path_terms | symbol_terms) & issue_terms
        if not exact and not overlap:
            continue
        selector = name
        if len(functions) > POLICY["large_file_test_functions"]:
            nodes = sorted(
                ((weight(topic_tokens(node)), node) for node in functions),
                key=lambda pair: (-pair[0], pair[1]),
            )
            if not nodes or nodes[0][0] <= 0:
                continue
            selector = f"{name}::{nodes[0][1]}"
        score = 1000 * exact + 4 * weight(path_terms) + 2 * weight(symbol_terms - path_terms)
        results.append({
            "selector": selector, "file": name, "score": score,
            "exact_filename_mention": exact, "matched_terms": sorted(overlap),
        })
    results.sort(key=lambda item: (-item["score"], item["selector"]))
    return results[:POLICY["max_selectors"]]


def visible_definition_paths(root):
    paths = set(RICH_OPTIONAL_TEST_DEFINITIONS)
    paths.update(name for name in CONFIG_FILES if (root / name).is_file())
    for path in (root / "tests").rglob("*"):
        relative = path.relative_to(root)
        if (any(part in EXCLUDED_DIRS for part in relative.parts)
                or path.name in EXCLUDED_NAMES or path.suffix in EXCLUDED_SUFFIXES):
            continue
        if path.is_file():
            paths.add(relative.as_posix())
    return tuple(sorted(paths))


def visible_definition_binding(root, files):
    regular, links = [], {}
    for name in files:
        path = root / name
        if not path.is_symlink():
            regular.append(name)
            continue
        target = path.resolve(strict=True)
        if not target.is_relative_to(root) or not target.is_file():
            raise CPUHold("UNSAFE_VISIBLE_DEFINITION_LINK", f"Invalid linked public fixture: {name}")
        links[name] = {"link_text": os.readlink(path), "target": target.relative_to(root).as_posix(),
                       "target_sha256": sha256_file(target)}
    digest = definition_files_fingerprint(root, regular, allow_missing=True)
    if links:
        digest = canonical_hash({"regular_sha256": digest, "internal_fixture_links": links})
    return digest, links

## 6. Qualify the selected baseline check

Normalize pytest node IDs to the checkout root after checking their collected file paths. Match the resulting node list to the baseline run and reconcile JUnit counts. A complete baseline PASS or FAIL can earn `CPU_READY`. Collection failures, test setup errors, timeouts and mismatches produce a recorded hold.


In [ ]:
CAPTURE_PLUGIN = """import json
import os
from pathlib import Path
import sys

def pytest_collection_finish(session):
    missing = set()
    if os.environ["GP03_PUBLIC_PACKAGE"] == "requests":
        for item in session.items:
            fixture_info = getattr(item, "_fixtureinfo", None)
            if fixture_info is None:
                continue
            for name in {"httpbin", "httpbin_secure"}.intersection(item.fixturenames):
                definitions = fixture_info.name2fixturedefs.get(name) or ()
                # Requests may wrap the plugin fixture by requesting its own name.
                if not any(name not in definition.argnames for definition in definitions):
                    missing.add(name)
    capture = {
        "root": str(session.config.rootpath.resolve()),
        "nodeids": [item.nodeid for item in session.items],
        "item_files": [str(item.path.resolve()) for item in session.items],
        "missing_optional_fixtures": sorted(missing),
    }
    Path(os.environ["GP03_CAPTURE_PATH"]).write_text(json.dumps(capture), encoding="utf-8")

def pytest_sessionfinish(session, exitstatus):
    target = Path(os.environ["GP03_CAPTURE_PATH"])
    if target.is_file():
        capture = json.loads(target.read_text(encoding="utf-8"))
        package = sys.modules.get(os.environ["GP03_PUBLIC_PACKAGE"])
        capture["production_origin"] = getattr(package, "__file__", None)
        target.write_text(json.dumps(capture), encoding="utf-8")
"""


def diagnostic_text(output):
    fragments = []
    for name in ("stdout.txt", "stderr.txt"):
        path = output / name
        with path.open("rb") as stream:
            head = stream.read(32 * 1024)
            if path.stat().st_size > len(head):
                stream.seek(max(len(head), path.stat().st_size - 32 * 1024))
                head += b"\n" + stream.read()
        fragments.append(head.decode("utf-8", errors="replace"))
    return "\n".join(fragments)


def missing_imports(text):
    return set(re.findall(r"ModuleNotFoundError: No module named ['\"]([A-Za-z0-9_.]+)['\"]", text))


def read_node_capture(path, selector, root, package_dir):
    if not path.is_file():
        raise CPUHold("NODE_CAPTURE_MISSING", "pytest produced no independent node ID capture")
    capture = json.loads(path.read_text(encoding="utf-8"))
    nodes = capture.get("nodeids", [])
    if not nodes:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    if len(nodes) > POLICY["max_collected_nodes"]:
        raise CPUHold("COLLECTION_LIMIT", f"Collected {len(nodes)} nodes, exceeding the frozen CPU cap")
    pytest_root = Path(capture["root"]).resolve(strict=True)
    if not pytest_root.is_relative_to(root):
        raise CPUHold("COLLECTION_IDENTITY", "pytest root is outside the staged checkout")
    files = capture.get("item_files", [])
    if len(files) != len(nodes):
        raise CPUHold("COLLECTION_IDENTITY", "pytest omitted a collected file identity")
    canonical = []
    for node, item_file in zip(nodes, files):
        if not isinstance(node, str):
            raise CPUHold("COLLECTION_IDENTITY", "Invalid pytest node ID")
        filename, delimiter, suffix = node.partition("::")
        relative = PurePosixPath(filename)
        if not delimiter or not suffix or relative.is_absolute() or ".." in relative.parts or "\\" in filename:
            raise CPUHold("COLLECTION_IDENTITY", "Invalid relative pytest node path")
        path_from_node = pytest_root / relative
        if (not path_from_node.is_relative_to(root) or str(path_from_node.resolve(strict=True)) != item_file):
            raise CPUHold("COLLECTION_IDENTITY", "Node path disagrees with the file collected by pytest")
        canonical.append(path_from_node.relative_to(root).as_posix() + "::" + suffix)
    if "::" in selector:
        permitted = lambda item: item == selector or item.startswith(selector + "[")
    else:
        permitted = lambda item: item.startswith(selector + "::")
    if not all(permitted(n) for n in canonical) or len(canonical) != len(set(canonical)):
        raise CPUHold("COLLECTION_IDENTITY", "Canonical node identities differ from the selector")
    origin = capture.get("production_origin")
    if origin and not Path(origin).resolve().is_relative_to(root / package_dir):
        raise CPUHold("PACKAGE_ORIGIN_CHANGED", "pytest imported production code outside the snapshot")
    return canonical, capture["missing_optional_fixtures"]


def run_visible_phase(*, selector, root, package, package_dir, temporary, output,
                      deadline, baseline_sha, collect_only):
    argv = [sys.executable, "-m", "pytest", selector, "-q", "--tb=short",
            "-p", "no:cacheprovider", "-p", "_gp03_capture"]
    capture = output / "nodeids.json"
    if collect_only:
        argv.append("--collect-only")
        cap = POLICY["collect_wall_seconds"]
    else:
        argv.append(f"--junitxml={output / 'junit.xml'}")
        cap = POLICY["baseline_wall_seconds"]
    result = bounded_command(
        argv, root, command_env(root, temporary, capture, package),
        output, remaining(deadline, cap),
    )
    result["source_sha256_after"] = WorkspaceFingerprinter(root).snapshot().sha256
    write_json(output / "command.json", result)
    if result["source_sha256_after"] != baseline_sha:
        raise CPUHold("SOURCE_CHANGED", "The selected pytest command changed the staged source or test definitions")
    if result["timed_out"]:
        reason = "COLLECTION_TIMEOUT" if collect_only else "BASELINE_TIMEOUT"
        raise CPUHold(reason, "The selected public pytest command exceeded its CPU limit")
    if collect_only and result["exit_code"] == 5:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    return result, capture


def choose_by_collection(*, candidates, root, package, package_dir, temporary, output,
                         deadline, baseline_sha, task_result):
    attempts = []
    task_result["collection_attempts"] = attempts
    for number, item in enumerate(candidates, 1):
        selector = item["selector"]
        attempt = {"selector": selector, "status": "COLLECTION_HOLD"}
        attempts.append(attempt)
        print(f"  collect {number}/{len(candidates)}  {selector}", flush=True)
        phase = output / f"collect_{number:02d}"
        result, capture = run_visible_phase(
            selector=selector, root=root, package=package, package_dir=package_dir,
            temporary=temporary, output=phase, deadline=deadline,
            baseline_sha=baseline_sha, collect_only=True,
        )
        attempt.update(exit_code=result["exit_code"], seconds=result["seconds"])
        if result["exit_code"] != 0:
            missing = missing_imports(diagnostic_text(phase))
            roots = {name.split(".")[0] for name in missing}
            attempt["missing_imports"] = sorted(missing)
            if result["exit_code"] == 2 and roots and roots.issubset(POLICY["optional_import_fallback"]):
                attempt["status"] = "OPTIONAL_IMPORT_FALLBACK"
                print(f"    missing {', '.join(sorted(missing))}", flush=True)
                continue
            reason = "COLLECTION_DEPENDENCY" if missing else "COLLECTION_ERROR"
            raise CPUHold(reason, f"Collection exit {result['exit_code']} for {selector}. See collection logs")
        nodes, unavailable = read_node_capture(capture, selector, root, package_dir)
        if unavailable:
            if not set(unavailable).issubset(POLICY["optional_fixture_fallback"]):
                raise CPUHold("COLLECTION_FIXTURE_IDENTITY", "Unexpected optional fixture diagnostic")
            attempt.update(status="OPTIONAL_FIXTURE_FALLBACK", missing_fixtures=unavailable)
            print(f"    unavailable fixture {', '.join(unavailable)}", flush=True)
            continue
        attempt.update(status="SELECTED", node_count=len(nodes))
        return selector, nodes
    raise CPUHold("OPTIONAL_DEPENDENCIES_UNAVAILABLE", "Every attempted selector needed an unavailable import or fixture")


def observe_baseline(*, selector, nodes, root, package, package_dir, temporary,
                     output, deadline, baseline_sha, task_result):
    phase = output / "baseline"
    print(f"  run {len(nodes)} collected node(s)  {selector}", flush=True)
    result, capture = run_visible_phase(
        selector=selector, root=root, package=package, package_dir=package_dir,
        temporary=temporary, output=phase, deadline=deadline,
        baseline_sha=baseline_sha, collect_only=False,
    )
    task_result.update(exit_code=result["exit_code"], baseline_seconds=result["seconds"])
    junit = phase / "junit.xml"
    if not junit.is_file():
        raise CPUHold("JUNIT_MISSING", "The baseline run produced no JUnit report")
    try:
        total, passed, failed, errors, skipped = junit_counts(junit)
    except (ET.ParseError, ValueError) as exc:
        raise CPUHold("JUNIT_INVALID", "The baseline JUnit report is malformed or contradictory") from exc
    task_result.update(actual_collected=total, passed=passed, failed=failed, errors=errors, skipped=skipped)
    if errors:
        raise CPUHold("BASELINE_SETUP_OR_TEARDOWN_ERROR", "JUnit recorded a test setup or teardown error")
    actual_nodes, unavailable = read_node_capture(capture, selector, root, package_dir)
    if unavailable or nodes != actual_nodes or total != len(nodes):
        raise CPUHold("BASELINE_NODE_MISMATCH", "Baseline execution differs from the independently collected nodes")
    if total == skipped:
        raise CPUHold("ALL_SKIPPED", "Every collected test was skipped")
    missing = missing_imports(diagnostic_text(phase)) if failed else set()
    if missing:
        task_result["missing_runtime_imports"] = sorted(missing)
        raise CPUHold("BASELINE_DEPENDENCY", "A baseline failure required a missing runtime import")
    if result["exit_code"] == 0 and passed > 0 and failed == 0:
        outcome = "PASS"
    elif result["exit_code"] == 1 and failed > 0:
        outcome = "FAIL"
    else:
        raise CPUHold("BASELINE_EXIT_MISMATCH", "Exit code and completed JUnit cases do not agree")
    reserve = max(15.0, round(result["seconds"] * 1.5 + 3, 1))
    if reserve >= POLICY["final_reserve_seconds"]:
        raise CPUHold("FINAL_RESERVE_LIMIT", "Measured baseline exceeds the planned final-gate reserve")
    return outcome, reserve, sha256_file(junit)


IMPORT_PROBE = """
import importlib
from pathlib import Path
import sys
module = importlib.import_module(sys.argv[1])
origin = Path(module.__file__).resolve()
print(origin)
if not origin.is_relative_to(Path(sys.argv[2]).resolve()):
    raise RuntimeError("Production package import escaped the public snapshot")
"""


def qualify_task(candidate, issue, competition, output, environment_sha):
    started = time.monotonic()
    deadline = started + POLICY["task_wall_seconds"]
    task = candidate["task_id"]
    row = {
        **candidate, "status": "CPU_HOLD", "reason": None,
        "expected_collected": None, "actual_collected": None, "baseline_outcome": None,
    }
    try:
        archive = competition / "snapshots" / f"{task}.tgz"
        if not archive.is_file():
            raise CPUHold("SNAPSHOT_MISSING", f"Expected organizer snapshot {archive.name}")
        if archive.stat().st_size != candidate["snapshot_bytes"]:
            raise CPUHold("SNAPSHOT_SIZE_MISMATCH", "Snapshot size differs from the GP00 inventory")
        row["snapshot_sha256"] = sha256_file(archive)
        with tempfile.TemporaryDirectory(prefix=f"gp03_{task}_") as scratch:
            temporary = Path(scratch).resolve()
            root, package, package_dir, links = stage_public_snapshot(
                archive, temporary / "checkout", candidate["repo"], deadline,
            )
            row["snapshot_link_count"] = len(links)
            if links:
                write_json(output / "Snapshot_Links.json", links)
            env = command_env(root, temporary)
            row.update(
                native_head=git_bytes(root, env, deadline, "rev-parse", "--verify", "HEAD^{commit}").decode().strip(),
                native_tree=git_bytes(root, env, deadline, "rev-parse", "--verify", "HEAD^{tree}").decode().strip(),
                production_root=package_dir + "/",
            )
            if not GIT_ID.fullmatch(row["native_head"]) or not GIT_ID.fullmatch(row["native_tree"]):
                raise CPUHold("NATIVE_GIT_IDENTITY", "Expected native Git commit and tree SHA-1 identities")
            tracked = git_bytes(root, env, deadline, "ls-files", "-z", "--", "tests").decode().split("\0")
            selectors = rank_public_selectors(root, tracked, issue)
            write_json(output / "Public_Selector_Plan.json", {
                "task_id": task, "rule": POLICY["selector_rule"],
                "public_issue_sha256": hashlib.sha256(issue.encode()).hexdigest(), "selectors": selectors,
            })
            if not selectors:
                raise CPUHold("NO_ISSUE_LINKED_TEST", "No bounded public pytest selection matched the issue")
            prepare_pytest_config(root)
            files = visible_definition_paths(root)
            definition_sha, fixture_links = visible_definition_binding(root, files)
            baseline_sha = WorkspaceFingerprinter(root).snapshot().sha256
            row.update(definition_files_sha256=definition_sha, baseline_source_sha256=baseline_sha)
            probe = bounded_command(
                [sys.executable, "-c", IMPORT_PROBE, package, str(root / package_dir)],
                root, env, output / "package_origin", remaining(deadline, 30),
            )
            if probe["timed_out"] or probe["exit_code"]:
                raise CPUHold("PACKAGE_IMPORT_HOLD", "The snapshot package failed its isolated import probe")
            if WorkspaceFingerprinter(root).snapshot().sha256 != baseline_sha:
                raise CPUHold("SOURCE_CHANGED", "The package import changed staged source or test definitions")
            (temporary / "_gp03_capture.py").write_text(CAPTURE_PLUGIN, encoding="utf-8")
            selector, nodes = choose_by_collection(
                candidates=selectors, root=root, package=package, package_dir=package_dir,
                temporary=temporary, output=output, deadline=deadline,
                baseline_sha=baseline_sha, task_result=row,
            )
            row.update(selector=selector, expected_collected=len(nodes))
            outcome, reserve, junit_sha = observe_baseline(
                selector=selector, nodes=nodes, root=root, package=package, package_dir=package_dir,
                temporary=temporary, output=output, deadline=deadline,
                baseline_sha=baseline_sha, task_result=row,
            )
            remaining(deadline, 1)
            command = [sys.executable, "-m", "pytest", selector, "-q", "--tb=short"]
            check = {
                "check_id": f"gp03_{task}", "origin": "visible_repo", "command": command,
                "working_dir": ".", "test_selection": selector, "expected_collected": len(nodes),
                "definition_files": list(files), "definition_files_sha256": definition_sha,
                "definition_sha256": canonical_hash({
                    "task": task, "argv": command, "definition_files_sha256": definition_sha,
                }),
                "allow_missing_definition_files": True, "pinned_environment_sha256": environment_sha,
                "reserve_seconds": reserve,
            }
            manifest = {
                "schema": "gp03.public-cpu-check.v1", "task_id": task, "repo": row["repo"],
                "base_commit": row["base_commit"], "native_head": row["native_head"],
                "native_tree": row["native_tree"], "snapshot_sha256": row["snapshot_sha256"],
                "production_root": row["production_root"], "baseline_source_sha256": baseline_sha,
                "source_policy": "gp02c_workspace_all_files_excluding_fixed_generated_artifacts",
                "definition_link_policy": "gp03_internal_fixture_links_v1" if fixture_links else "regular_files_v1",
                "definition_link_targets": fixture_links,
                "environment_sha256": environment_sha, "expected_nodeids": nodes,
                "expected_nodeids_sha256": canonical_hash(nodes), "baseline_junit_sha256": junit_sha,
                "baseline_outcome": outcome, "check": check,
            }
            path = output / "CPU_Check_Manifest.json"
            write_json(path, manifest)
            row.update(status="CPU_READY", baseline_outcome=outcome, reserve_seconds=reserve,
                       manifest_sha256=sha256_file(path))
    except CPUHold as exc:
        row.update(reason=exc.reason, detail=str(exc))
    except (OSError, ValueError, tarfile.TarError, subprocess.SubprocessError) as exc:
        row.update(reason="INPUT_OR_RUNTIME_ERROR", detail=f"{type(exc).__name__}: {str(exc)[:500]}")
    row["elapsed_seconds"] = round(time.monotonic() - started, 3)
    row["environment_sha256"] = environment_sha
    write_json(output / "Task_Result.json", row)
    return row

## 7. Verify previous receipts, checkpoint and export evidence

In [ ]:
def artifact_paths(folder, excluded=()):
    files = {}
    for path in sorted(folder.rglob("*")):
        relative = path.relative_to(folder).as_posix()
        if path.is_symlink():
            raise CPUHold("UNSAFE_EVIDENCE_PATH", f"Symlink in CPU evidence at {relative}")
        if path.is_file() and relative not in excluded:
            files[relative] = path
    return files


def artifact_index(files):
    return {name: {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
            for name, path in files.items()}


def read_checkpoint(folder, candidate, competition, run_sha):
    receipt = json.loads((folder / "Task_Receipt.json").read_text(encoding="utf-8"))
    if (receipt["run_sha256"] != run_sha or receipt["task_id"] != candidate["task_id"]
            or receipt["files"] != artifact_index(artifact_paths(folder, ("Task_Receipt.json",)))):
        raise CPUHold("CHECKPOINT_IDENTITY", f"Saved evidence failed validation for {candidate['task_id']}")
    row = json.loads((folder / "Task_Result.json").read_text(encoding="utf-8"))
    if any(row[key] != value for key, value in candidate.items()):
        raise CPUHold("CHECKPOINT_CANDIDATE", f"Saved candidate identity differs for {candidate['task_id']}")
    if row.get("snapshot_sha256"):
        archive = competition / "snapshots" / f"{candidate['task_id']}.tgz"
        if not archive.is_file() or sha256_file(archive) != row["snapshot_sha256"]:
            raise CPUHold("CHECKPOINT_SNAPSHOT", f"Mounted snapshot differs for {candidate['task_id']}")
    return row


def open_previous_evidence(spec, explicit_path, working, candidates, release_sha, environment_sha):
    if explicit_path is not None:
        path = Path(explicit_path)
    else:
        path = working / spec["name"]
        if not path.is_file():
            path = next(Path("/kaggle/input").rglob(spec["name"]), path)
    if not path.is_file():
        raise CPUHold("PRIOR_EVIDENCE_MISSING",
                      f"Keep {spec['name']} in /kaggle/working or add it as a Kaggle input")
    if sha256_file(path) != spec["sha256"]:
        raise CPUHold("PRIOR_EVIDENCE_SHA256", "The previous qualification ZIP differs from the verified v1.0 archive")
    members = verify_zip(path)
    with zipfile.ZipFile(path) as archive:
        prior = json.loads(archive.read("GP03_CPU_Report.json"))
        roster = json.loads(archive.read("GP03_Provisional_Candidates.json"))
    if (prior["notebook_recipe_sha256"] != spec["recipe"] or prior["tasks_sha256"] != release_sha
            or prior["gp00_split_sha256"] != GP00_SPLIT_SHA256 or roster["candidates"] != list(candidates)
            or prior["pending"] or prior["model_calls"] or prior["protected_verifier_calls"]
            or prior["held_out_agent_attempts"] or prior["evaluation_frozen"]
            or canonical_hash(prior["run_identity"]) != prior["run_sha256"]):
        raise CPUHold("PRIOR_EVIDENCE_IDENTITY", "The original report does not match the pinned CPU-only qualification")
    if prior["environment_sha256"] != environment_sha:
        raise CPUHold("RETRY_ENVIRONMENT_MISMATCH", "Use the same Kaggle CPU runtime and installed dependencies as v1.0")
    rows = {row["task_id"]: row for row in prior["rows"]}
    if (len(rows) != len(prior["rows"]) or set(rows) != {candidate["task_id"] for candidate in candidates}
            or any(any(rows[c["task_id"]].get(k) != v for k, v in c.items()) for c in candidates)):
        raise CPUHold("PRIOR_EVIDENCE_CANDIDATES", "The original per-task result set differs from GP00")
    retained = {task for task, row in rows.items()
                if row["status"] == "CPU_READY" or row["reason"] == "NO_ISSUE_LINKED_TEST"}
    return path, prior, rows, retained, members


def stage_retained_task(previous, target, candidate, competition):
    path, prior, rows, _retained, members = previous
    task = candidate["task_id"]
    if not target.exists():
        prefix = f"tasks/{task}/"
        with tempfile.TemporaryDirectory(prefix=".gp03_retaining_", dir=target.parent) as staging:
            pending = Path(staging) / "result"
            pending.mkdir()
            with zipfile.ZipFile(path) as archive:
                for name in members:
                    if name.startswith(prefix):
                        destination = pending / name.removeprefix(prefix)
                        destination.parent.mkdir(parents=True, exist_ok=True)
                        destination.write_bytes(archive.read(name))
            row = read_checkpoint(pending, candidate, competition, prior["run_sha256"])
            if row != rows[task]:
                raise CPUHold("RETAINED_ROW_MISMATCH", f"Previous task receipt differs from the report for {task}")
            pending.rename(target)
            return row
    row = read_checkpoint(target, candidate, competition, prior["run_sha256"])
    if row != rows[task]:
        raise CPUHold("RETAINED_ROW_MISMATCH", f"Previous task receipt differs from the report for {task}")
    return row


def run_cpu_cohort(competition, working, label, candidates, development, release_sha, *,
                   prior_spec=None, prior_path=None):
    started = time.monotonic()
    working.mkdir(parents=True, exist_ok=True)
    report = {
        "schema": "gp03.cpu-qualification.v1", "status": "GP03_CPU_INPUT_HOLD",
        "policy": POLICY, "tasks_sha256": release_sha, "gp00_split_sha256": GP00_SPLIT_SHA256,
        "notebook_recipe_sha256": NOTEBOOK_RECIPE_SHA256,
        "candidate_count": len(candidates), "model_calls": 0, "protected_verifier_calls": 0,
        "held_out_agent_attempts": 0, "evaluation_frozen": False, "rows": [],
    }
    archived = {}
    retained = set()
    try:
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", label):
            raise CPUHold("RUN_LABEL_INVALID", "Use a short run label with letters, numbers, underscore or hyphen")
        issues = load_gp03_inputs(competition, candidates, development, release_sha)
        profile = environment_profile()
        environment_sha = environment_fingerprint(profile)
        previous = None
        if prior_spec is not None:
            previous = open_previous_evidence(
                prior_spec, prior_path, working, candidates, release_sha, environment_sha,
            )
            retained = previous[3]
            report["prior_evidence"] = {
                "archive_sha256": prior_spec["sha256"], "archive_name": prior_spec["name"],
                "run_sha256": previous[1]["run_sha256"],
                "retained_task_ids": sorted(retained),
                "retry_task_ids": [c["task_id"] for c in candidates if c["task_id"] not in retained],
            }
        identity = {
            "schema": report["schema"], "notebook_recipe_sha256": NOTEBOOK_RECIPE_SHA256,
            "policy": POLICY, "tasks_sha256": release_sha, "gp00_split_sha256": GP00_SPLIT_SHA256,
            "candidate_sha256": canonical_hash(candidates), "environment_sha256": environment_sha,
            "mount": str(competition.resolve()), "run_label": label,
            "prior_evidence_sha256": prior_spec["sha256"] if prior_spec else None,
        }
        run_sha = canonical_hash(identity)
        report.update(environment=profile, environment_sha256=environment_sha, run_identity=identity,
                      run_sha256=run_sha)
        run_root = working / "GP03_CPU_Checkpoints" / f"{label}_{run_sha[:16]}"
        run_root.mkdir(parents=True, exist_ok=True)
        print(f"{len(candidates)} provisional candidates | pytest {profile['pytest_version']} | model calls 0",
              flush=True)
        if previous is not None:
            print(f"Retaining {len(retained)} original results | rechecking {len(candidates) - len(retained)} holds",
                  flush=True)
        print("Checkpoint directory", run_root, flush=True)
        with (run_root / "run.lock").open("a") as lock:
            try:
                fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError as exc:
                raise CPUHold("COHORT_ALREADY_RUNNING", "This GP03 checkpoint directory has an active run") from exc
            identity_file = run_root / "Run_Identity.json"
            if identity_file.exists():
                if json.loads(identity_file.read_text()) != identity:
                    raise CPUHold("RUN_IDENTITY_MISMATCH", "Use a fresh run label for a different qualification run")
            else:
                write_json(identity_file, identity)
            tasks = run_root / "tasks"
            tasks.mkdir(exist_ok=True)
            retained_tasks = run_root / "retained_v1_0"
            if previous is not None:
                retained_tasks.mkdir(exist_ok=True)
            for index, candidate in enumerate(candidates, 1):
                task = candidate["task_id"]
                target = tasks / task
                prefix = f"[{index:02d}/{len(candidates):02d}] {task}"
                if task in retained:
                    target = retained_tasks / task
                    print(f"{prefix} verifying retained v1.0 evidence", flush=True)
                    row = stage_retained_task(previous, target, candidate, competition)
                elif target.exists():
                    print(f"{prefix} verifying saved checkpoint", flush=True)
                    row = read_checkpoint(target, candidate, competition, run_sha)
                else:
                    print(f"{prefix} staging public snapshot", flush=True)
                    with tempfile.TemporaryDirectory(prefix=".gp03_pending_", dir=tasks) as staging:
                        pending = Path(staging) / "result"
                        pending.mkdir()
                        row = qualify_task(candidate, issues[task], competition, pending, environment_sha)
                        write_json(pending / "Task_Receipt.json", {
                            "schema": "gp03.cpu-task-receipt.v1", "run_sha256": run_sha,
                            "task_id": task, "files": artifact_index(artifact_paths(pending)),
                        })
                        pending.rename(target)
                report["rows"].append(row)
                archived.update({f"tasks/{task}/{name}": path for name, path in artifact_paths(target).items()})
                detail = row["baseline_outcome"] or row["reason"]
                print(f"{prefix} {row['status']} | {detail} | {row['elapsed_seconds']:.1f}s", flush=True)
        ready = sum(row["status"] == "CPU_READY" for row in report["rows"])
        report["status"] = (
            "GP03_CPU_QUALIFIED" if ready == len(candidates)
            else "GP03_CPU_PARTIAL" if ready else "GP03_CPU_HOLD"
        )
    except KeyboardInterrupt:
        report["status"] = "GP03_CPU_INTERRUPTED"
        report["detail"] = "The interrupted task can resume from the last sealed checkpoint"
    except (CPUHold, OSError, ValueError, KeyError) as exc:
        report["status"] = "GP03_CPU_INPUT_HOLD"
        report["reason"] = exc.reason if isinstance(exc, CPUHold) else "COHORT_INPUT_OR_CHECKPOINT_ERROR"
        report["detail"] = str(exc)[:700]

    counts = Counter(row["status"] for row in report["rows"])
    report.update(
        cpu_ready=counts["CPU_READY"], cpu_hold=counts["CPU_HOLD"],
        pending=len(candidates) - len(report["rows"]), elapsed_seconds=round(time.monotonic() - started, 3),
        retained_results=sum(row["task_id"] in retained for row in report["rows"]),
    )
    report["by_repository"] = {}
    for repo in sorted({candidate["repo"] for candidate in candidates}):
        group = [row for row in report["rows"] if row["repo"] == repo]
        planned = sum(candidate["repo"] == repo for candidate in candidates)
        report["by_repository"][repo] = {
            "planned": planned, "ready": sum(row["status"] == "CPU_READY" for row in group),
            "hold": sum(row["status"] == "CPU_HOLD" for row in group), "pending": planned - len(group),
        }
    return report, archived


def export_cpu_evidence(report, archived, candidates, working):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    name = f"GP03_CPU_Qualification_{timestamp}_{uuid.uuid4().hex[:8]}"
    with tempfile.TemporaryDirectory(prefix=".gp03_export_", dir=working) as scratch:
        output = Path(scratch)
        write_json(output / "GP03_CPU_Report.json", report)
        write_json(output / "GP03_Provisional_Candidates.json", {
            "gp00_split_sha256": GP00_SPLIT_SHA256, "candidates": candidates,
        })
        fields = (
            "task_id", "repo", "base_commit", "qualification_origin", "status", "reason", "selector", "baseline_outcome",
            "expected_collected", "actual_collected", "passed", "failed", "errors", "skipped",
            "baseline_seconds", "elapsed_seconds", "reserve_seconds", "manifest_sha256",
        )
        observed = {row["task_id"]: row for row in report["rows"]}
        retained = set(report.get("prior_evidence", {}).get("retained_task_ids", []))
        with (output / "GP03_Eligibility.csv").open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
            writer.writeheader()
            for candidate in candidates:
                task = candidate["task_id"]
                row = dict(observed.get(task, {**candidate, "status": "PENDING"}))
                if task in observed:
                    row["qualification_origin"] = "RETAINED_V1_0" if task in retained else "V1_1"
                writer.writerow(row)
        files = {**artifact_paths(output), **archived}
        return seal_zip(working / (name + ".zip"), files)

## 8. Retry the affected cases

The ZIP contains the combined eligibility CSV, original and new per-task receipts, public pytest logs, runtime profile and checksum manifest. Use the eligible cases and Kaggle budget when freezing the final paired B/C cohort.


In [ ]:
GP03_REPORT, GP03_ARTIFACTS = run_cpu_cohort(
    COMPETITION_DIR, WORK_DIR, RUN_LABEL, GP03_CANDIDATES, GP03_DEVELOPMENT, TASKS_SHA256,
    prior_spec=GP03_PRIOR, prior_path=PRIOR_CPU_ZIP,
)
GP03_EVIDENCE_ZIP = export_cpu_evidence(GP03_REPORT, GP03_ARTIFACTS, GP03_CANDIDATES, WORK_DIR)
print(
    f"{GP03_REPORT['status']} | {GP03_REPORT['cpu_ready']}/{GP03_REPORT['candidate_count']} CPU-ready"
    f" | {GP03_REPORT['cpu_hold']} holds | {GP03_REPORT['pending']} pending"
    f" | model calls 0 | elapsed {GP03_REPORT['elapsed_seconds'] / 60:.1f} min",
    flush=True,
)
for repo, counts in GP03_REPORT["by_repository"].items():
    print(f"  {repo:<18} ready {counts['ready']:>2}/{counts['planned']:<2}  "
          f"hold {counts['hold']:>2}  pending {counts['pending']:>2}", flush=True)
if GP03_REPORT.get("detail"):
    print(GP03_REPORT["detail"], flush=True)
offer_zip_download(GP03_EVIDENCE_ZIP, auto_download=AUTO_DOWNLOAD_ZIP)